# FinTune-Sentinel v2: training on Kaggle (free GPU)

**Before running:**
1. Settings → Accelerator → **GPU T4 x2**; Settings → **Internet on**
2. Add Input → attach these 3 datasets:
   - `yousefsaeedian/financial-q-and-a-10k`
   - `yousefsaeedian/finance-llama2-1k`
   - `visalakshiiyer/question-answering-financial-data`
3. Add-ons → Secrets → add `HF_TOKEN` (Hugging Face token with **write** access)

**Order:** run steps 1–3 (smoke test, ~20 min). Only when that works, run step 4 with *Save Version → Save & Run All* so it keeps running in the background.

## 1. Code and dependencies

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Unsloth trains on one GPU
%cd /kaggle/working
!git clone -b v2-pipeline https://github.com/Prakruthi19/FinTune-Sentinel.git
%cd FinTune-Sentinel
!pip install -q unsloth

## 2. Data

In [ ]:
!mkdir -p data
!find /kaggle/input -type f \( -name "*.csv" -o -name "*.json" \) -exec cp {} data/ \;
!ls -la data   # expect Financial-QA-10k.csv, finance-llama2-1k.csv, train.json, dev.json, test.json

## 3. Smoke test: 50 examples per source, 20 steps, no GGUF export (~20 min)
If this fails, fix it before spending hours on step 4.

In [ ]:
!python -m scripts.train_sft --load-in-4bit --batch-size 2 --max-len 2048     --max-samples 50 --max-steps 20 --skip-gguf --out runs/smoke

## 4. Full training (~3–6 h). Run via *Save Version → Save & Run All*

In [ ]:
!python -m scripts.train_sft --load-in-4bit --batch-size 2 --max-len 2048 --epochs 1 --out runs/v2
!cat runs/v2/metrics.json
!find runs/v2 -name "*.gguf" -exec ls -lh {} \;

## 5. Plot training vs eval loss

In [ ]:
import json, matplotlib.pyplot as plt
log = json.load(open("runs/v2/log_history.json"))
tr = [(r["step"], r["loss"]) for r in log if "loss" in r]
ev = [(r["step"], r["eval_loss"]) for r in log if "eval_loss" in r]
plt.plot(*zip(*tr), label="train"); plt.plot(*zip(*ev), "o-", label="eval")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("FinTune-Sentinel v2")
plt.savefig("runs/v2/loss.png", dpi=150); plt.show()

## 6. Upload to Hugging Face

In [ ]:
import glob
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
repo = "Prakruthi/FinTune-Sentinel-v2-GGUF"
api.create_repo(repo, exist_ok=True)
for f in glob.glob("runs/v2/**/*.gguf", recursive=True) + ["runs/v2/metrics.json", "runs/v2/loss.png"]:
    print("uploading", f)
    api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=repo)